# RT-DETR-L — VINS / vins13 (Google Colab)
Thesis experiment: `rtdetr_l_vins_vins13`

**Before running:**
1. Upload `uidet-vins13.zip` and `uidet-code.zip` to your Google Drive under a folder called `uidet/`
2. Add your `WANDB_API_KEY` to Colab Secrets (🔑 icon in the left panel)
3. Set `TEST_RUN = True` for a 3-epoch timing test, `False` for the full run

In [ ]:
# ── CONFIG ──────────────────────────────────────────────────────────────────
TEST_RUN = True   # set False for full training run
DRIVE_FOLDER = 'uidet'  # folder in your Google Drive root containing the zips
# ────────────────────────────────────────────────────────────────────────────

In [ ]:
!pip install -q pycocotools wandb pyyaml ultralytics

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, sys, zipfile, shutil
from pathlib import Path

DRIVE_BASE = Path('/content/drive/MyDrive') / DRIVE_FOLDER
WORKING    = Path('/content/working')
WORKING.mkdir(exist_ok=True)

# Extract code zip
code_zip = DRIVE_BASE / 'uidet-code.zip'
code_dir = WORKING / 'code'
if not code_dir.exists():
    print('Extracting code...')
    with zipfile.ZipFile(code_zip) as z:
        z.extractall(code_dir)
print('Code extracted:', sorted(p.name for p in (code_dir / 'src').iterdir()))

# Extract data zip
data_zip  = DRIVE_BASE / 'uidet-vins13.zip'
data_dir  = WORKING / 'vins13_input'
if not data_dir.exists():
    print('Extracting data (this may take a minute)...')
    with zipfile.ZipFile(data_zip) as z:
        z.extractall(data_dir)
print('Data extracted:', sorted(p.name for p in data_dir.iterdir()))

sys.path.insert(0, str(code_dir / 'src'))
import uidet
print('uidet imported OK:', uidet.__file__)

In [ ]:
import yaml

actual_images      = data_dir / 'images'
actual_labels      = data_dir / 'labels'
actual_annotations = data_dir / 'annotations'

work_data = WORKING / 'vins13'
work_data.mkdir(parents=True, exist_ok=True)

# Symlink images and labels (large — avoids copying)
for name, src in [('images', actual_images), ('labels', actual_labels)]:
    link = work_data / name
    if link.is_symlink(): link.unlink()
    link.symlink_to(src)
    print(f'{name} link ok:', link.exists())

# Copy annotation JSONs
ann_dir = work_data / 'annotations'
ann_dir.mkdir(exist_ok=True)
for f in actual_annotations.glob('*.json'):
    shutil.copy2(f, ann_dir / f.name)
    print('copied:', f.name)

# Write corrected data.yaml
with open(data_dir / 'data.yaml') as f:
    data_cfg = yaml.safe_load(f)
data_cfg['path'] = str(work_data)
data_yaml_path = work_data / 'data.yaml'
with open(data_yaml_path, 'w') as f:
    yaml.safe_dump(data_cfg, f, sort_keys=False)
print('data.yaml written OK')
print(data_cfg)

In [ ]:
import wandb

try:
    from google.colab import userdata
    wandb.login(key=userdata.get('WANDB_API_KEY'))
    print('WandB logged in via Colab secret')
except Exception as e:
    print(f'Secret not found ({e}), falling back...')
    wandb.login()

In [ ]:
from uidet.models.base import TrainConfig, build_detector
from uidet.train import get_class_names

EXP_NAME   = 'rtdetr_l_vins_vins13'
MODEL_NAME = 'rtdetr_l'
DATASET    = 'vins'
TAXONOMY   = 'vins13'

out_dir = WORKING / 'results_v2' / EXP_NAME
out_dir.mkdir(parents=True, exist_ok=True)

class_names = get_class_names(data_yaml_path)
print(f'{len(class_names)} classes: {class_names}')

cfg = TrainConfig(
    name=EXP_NAME,
    out_dir=out_dir,
    data_yaml=data_yaml_path,
    coco_val_json=ann_dir / f'{DATASET}_{TAXONOMY}_val.json',
    coco_test_json=ann_dir / f'{DATASET}_{TAXONOMY}_test.json',
    epochs=3 if TEST_RUN else 60,
    batch=8,
    imgsz=640,
    lr0=0.0001,
    seed=42,
    device='0',
    workers=4,
    amp=True,
    extra={
        'patience': 999 if TEST_RUN else 15,  # disable early stopping for timing test
        'cos_lr': True,
    },
    wandb_project='uidet-thesis',
    wandb_entity=None,
    wandb_tags=['colab', 'rtdetr', DATASET, TAXONOMY] + (['test'] if TEST_RUN else []),
)
print(f'Config OK -> {out_dir}  ({"TEST RUN 3 epochs" if TEST_RUN else "FULL RUN 60 epochs"})')

In [ ]:
import json, time
from uidet.train import _wandb_init, _wandb_log_epochs_from_csv, _wandb_log_summary, _wandb_finish

cfg_dict = {
    'name': EXP_NAME, 'model': MODEL_NAME,
    'dataset': DATASET, 'taxonomy': TAXONOMY,
    'wandb_project': 'uidet-thesis', 'wandb_entity': None, 'wandb_tags': cfg.wandb_tags,
}

wandb_run = _wandb_init(cfg_dict, cfg, class_names) if not TEST_RUN else None

detector = build_detector(MODEL_NAME, num_classes=len(class_names), class_names=class_names)
print(f'Training {MODEL_NAME} on {DATASET}/{TAXONOMY}...')

t0 = time.perf_counter()
weights = detector.train(cfg)
elapsed = time.perf_counter() - t0

print(f'\nDone in {elapsed/60:.1f} min')
if TEST_RUN:
    print(f'Estimated time per epoch: {elapsed/60/3:.1f} min')
    print(f'Estimated full run (60 epochs): {elapsed/60/3*60:.0f} min  ({elapsed/60/3*60/60:.1f} h)')
else:
    _wandb_log_epochs_from_csv(out_dir, wandb_run)

In [ ]:
# Only run eval on a full training run
if not TEST_RUN:
    import time
    from uidet.utils.metrics import coco_evaluate, detections_to_coco, format_metrics_table

    detector.load(weights)

    gt_path = ann_dir / f'{DATASET}_{TAXONOMY}_test.json'
    gt      = json.loads(gt_path.read_text())
    items   = [(im['id'], work_data / im['file_name']) for im in gt['images']]
    name_to_cat_id = {c: i + 1 for i, c in enumerate(class_names)}

    predictions, t0 = [], time.perf_counter()
    for i in range(0, len(items), 4):
        chunk = items[i:i+4]
        batch_dets = detector.predict_batch([p for _, p in chunk], conf=0.001, iou=0.6)
        for image_id, dets in zip([iid for iid, _ in chunk], batch_dets):
            predictions.extend(detections_to_coco(dets, image_id, name_to_cat_id))
    dt = time.perf_counter() - t0

    eval_dir = out_dir / 'coco_eval_test'
    metrics  = coco_evaluate(str(gt_path), predictions, eval_dir)
    metrics['inference_seconds'] = dt
    metrics['inference_fps']     = len(items) / dt
    (eval_dir / 'metrics.json').write_text(json.dumps(metrics, indent=2))
    print(format_metrics_table(metrics))
else:
    print('Skipping eval — test run only.')

In [ ]:
if not TEST_RUN:
    if wandb_run is not None:
        wandb.log({
            'test/mAP':   metrics['mAP'],
            'test/mAP50': metrics['mAP50'],
            'test/mAP75': metrics['mAP75'],
            **{f'test/AP/{cls}':   v['AP']   for cls, v in metrics['per_class'].items()},
            **{f'test/AP50/{cls}': v['AP50'] for cls, v in metrics['per_class'].items()},
        })
        _wandb_log_summary(wandb_run, metrics, split='test')
        _wandb_finish(wandb_run)

    # Zip and save results back to Drive
    import shutil
    out_zip = DRIVE_BASE / f'{EXP_NAME}.zip'
    shutil.make_archive(str(WORKING / EXP_NAME), 'zip', str(out_dir))
    shutil.copy(str(WORKING / f'{EXP_NAME}.zip'), str(out_zip))
    print('Saved to Drive:', out_zip)
    print(json.dumps(metrics, indent=2))
else:
    print('Test run complete. Set TEST_RUN = False and re-run for full training.')